In [11]:
# ============================================
# 1. Instalar bibliotecas
# ============================================
!pip install -q rasterio matplotlib numpy

# ============================================
# 2. Importar bibliotecas
# ============================================
import os
import glob
import math
import numpy as np
import rasterio
import matplotlib.pyplot as plt
import matplotlib as mpl

from rasterio.crs import CRS
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.transform import from_origin
from google.colab import drive

In [2]:
# ============================================
# 3. Montar Google Drive
# ============================================
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
# ============================================
# 4. Caminhos principais
# ============================================
ROOT = "/content/drive/Shareddrives/Batimetrias_Babitonga"

AOIS = {
    "01_AOI01": "AOI 1",
    "02_AOI02": "AOI 2",
    "03_AOI03": "AOI 3"
}

OUTPUT_ROOT = os.path.join(ROOT, "_FIGURAS_SUPERFICIES")
os.makedirs(OUTPUT_ROOT, exist_ok=True)

TARGET_CRS = CRS.from_epsg(4326)


In [16]:
# ============================================
# 5. Função para encontrar TIFF por ano
# ============================================
def find_yearly_tifs(aoi_dir):
    """
    Retorna dict {ano: caminho_tif}
    Se houver mais de um tif no ano, usa o primeiro encontrado.
    """
    surf_dir = os.path.join(aoi_dir, "04_SUPERFICIES")
    yearly = {}

    if not os.path.isdir(surf_dir):
        return yearly

    year_folders = sorted(os.listdir(surf_dir))

    for year in year_folders:
        year_path = os.path.join(surf_dir, year)
        if not os.path.isdir(year_path):
            continue

        tif_list = sorted(glob.glob(os.path.join(year_path, "*.tif")) +
                          glob.glob(os.path.join(year_path, "*.tiff")))

        if len(tif_list) == 0:
            tif_list = sorted(glob.glob(os.path.join(year_path, "**", "*.tif"), recursive=True) +
                              glob.glob(os.path.join(year_path, "**", "*.tiff"), recursive=True))

        if tif_list:
            yearly[year] = tif_list[0]

    return yearly

# ============================================
# 6. Detectar CRS de fallback
# ============================================
def get_dominant_crs(paths):
    crs_list = []

    for p in paths:
        try:
            with rasterio.open(p) as src:
                if src.crs is not None:
                    crs_list.append(src.crs.to_string())
        except:
            pass

    if not crs_list:
        print("  [AVISO] Nenhum raster com CRS definido. Assumindo EPSG:4326")
        return TARGET_CRS

    from collections import Counter
    dominant = Counter(crs_list).most_common(1)[0][0]
    return CRS.from_string(dominant)

# ============================================
# 7. Construir grade comum em EPSG:4326
# ============================================
def build_common_grid_4326(yearly_tifs):
    """
    Cria uma grade comum por AOI em EPSG:4326,
    usando a união das extensões de todos os rasters.
    """
    paths = list(yearly_tifs.values())
    fallback_crs = get_dominant_crs(paths)

    bounds_all = []
    xres_list = []
    yres_list = []

    for path in paths:
        with rasterio.open(path) as src:
            src_crs = src.crs if src.crs is not None else fallback_crs

            # resolução aproximada original
            xres_list.append(abs(src.transform.a))
            yres_list.append(abs(src.transform.e))

            # bounds transformados para 4326
            if src_crs == TARGET_CRS:
                b = src.bounds
                bounds_all.append((b.left, b.bottom, b.right, b.top))
            else:
                b = transform_bounds(src_crs, TARGET_CRS, *src.bounds, densify_pts=21)
                bounds_all.append(b)

    # união total
    minx = min(b[0] for b in bounds_all)
    miny = min(b[1] for b in bounds_all)
    maxx = max(b[2] for b in bounds_all)
    maxy = max(b[3] for b in bounds_all)

    # resolução-alvo em graus:
    # aqui usamos a menor resolução encontrada para evitar perda grosseira
    xres = min(xres_list)
    yres = min(yres_list)

    width = int(math.ceil((maxx - minx) / xres))
    height = int(math.ceil((maxy - miny) / yres))

    transform = from_origin(minx, maxy, xres, yres)

    return {
        "crs": TARGET_CRS,
        "transform": transform,
        "width": width,
        "height": height,
        "fallback_crs": fallback_crs
    }


# ============================================
# 8. Ler e reprojetar raster para grade comum
# ============================================
def read_reprojected_to_grid(path, common_grid):
    with rasterio.open(path) as src:
        arr = src.read(1).astype("float32")
        src_nodata = src.nodata
        src_crs = src.crs if src.crs is not None else common_grid["fallback_crs"]

        if src_nodata is not None:
            arr[arr == src_nodata] = np.nan

        dst = np.full(
            (common_grid["height"], common_grid["width"]),
            np.nan,
            dtype="float32"
        )

        reproject(
            source=arr,
            destination=dst,
            src_transform=src.transform,
            src_crs=src_crs,
            dst_transform=common_grid["transform"],
            dst_crs=common_grid["crs"],
            src_nodata=np.nan,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear
        )

    return dst

# ============================================
# 9. Calcular min/max globais por AOI já na grade comum
# ============================================
def get_global_min_max_reprojected(yearly_tifs, common_grid):
    mins = []
    maxs = []

    for year, tif_path in yearly_tifs.items():
        arr = read_reprojected_to_grid(tif_path, common_grid)

        if np.all(np.isnan(arr)):
            continue

        mins.append(np.nanmin(arr))
        maxs.append(np.nanmax(arr))

    if not mins or not maxs:
        return None, None

    return float(np.min(mins)), float(np.max(maxs))



In [17]:
# ============================================
# 10. Salvar figura do DEM
# ============================================
def save_dem_figure(arr, vmin, vmax, title, out_path):
    fig, ax = plt.subplots(figsize=(6, 5), dpi=200)

    im = ax.imshow(arr, cmap="viridis", vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])

    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Elevação", fontsize=10)

    plt.tight_layout()
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

In [24]:
# ============================================
# 11. Salvar colorbar isolada
# ============================================
def save_colorbar_only(vmin, vmax, out_path, orientation="vertical"):
    cmap = mpl.cm.viridis
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)

    if orientation == "vertical":
        fig, ax = plt.subplots(figsize=(2, 6), dpi=200)
        fig.subplots_adjust(left=0.45, right=0.75)
    else:
        fig, ax = plt.subplots(figsize=(17, 1.5), dpi=200)
        fig.subplots_adjust(bottom=0.5, top=0.8)

    cbar = mpl.colorbar.ColorbarBase(
        ax,
        cmap=cmap,
        norm=norm,
        orientation=orientation
    )
    cbar.set_label("Elevação", fontsize=10)

    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close(fig)


In [25]:
# ============================================
# 12. Processar AOIs
# ============================================
for aoi_folder, aoi_label in AOIS.items():
    print(f"\nProcessando {aoi_label}...")

    aoi_dir = os.path.join(ROOT, aoi_folder)
    yearly_tifs = find_yearly_tifs(aoi_dir)

    if len(yearly_tifs) == 0:
        print(f"  [AVISO] Nenhum TIFF encontrado em {aoi_folder}/04_SUPERFICIES")
        continue

    # grade comum em 4326
    common_grid = build_common_grid_4326(yearly_tifs)

    # extremos globais por AOI já reprojetados
    vmin, vmax = get_global_min_max_reprojected(yearly_tifs, common_grid)

    if vmin is None or vmax is None:
        print(f"  [AVISO] Não foi possível calcular min/max para {aoi_label}")
        continue

    print(f"  Intervalo global {aoi_label}: min = {vmin:.3f}, max = {vmax:.3f}")

    out_aoi_dir = os.path.join(OUTPUT_ROOT, aoi_folder)
    os.makedirs(out_aoi_dir, exist_ok=True)

    # figuras anuais
    for year, tif_path in sorted(yearly_tifs.items()):
        arr = read_reprojected_to_grid(tif_path, common_grid)

        title = f"{aoi_label} - {year}"
        out_fig = os.path.join(out_aoi_dir, f"{aoi_label.replace(' ', '_')}_{year}.png")

        save_dem_figure(arr, vmin, vmax, title, out_fig)
        print(f"  [OK] Figura salva: {os.path.basename(out_fig)}")

    # colorbar vertical
    out_cb_v = os.path.join(out_aoi_dir, f"{aoi_label.replace(' ', '_')}_colorbar_vertical.png")
    save_colorbar_only(vmin, vmax, out_cb_v, orientation="vertical")

    # colorbar horizontal
    out_cb_h = os.path.join(out_aoi_dir, f"{aoi_label.replace(' ', '_')}_colorbar_horizontal_17.png")
    save_colorbar_only(vmin, vmax, out_cb_h, orientation="horizontal")

    print(f"  [OK] Colorbars salvas para {aoi_label}")

print("\nConcluído.")


Processando AOI 1...
  Intervalo global AOI 1: min = -1.795, max = 1.745
  [OK] Figura salva: AOI_1_1985.png
  [OK] Figura salva: AOI_1_1986.png
  [OK] Figura salva: AOI_1_1987.png
  [OK] Figura salva: AOI_1_1988.png
  [OK] Figura salva: AOI_1_1991.png
  [OK] Figura salva: AOI_1_1993.png
  [OK] Figura salva: AOI_1_1994.png
  [OK] Figura salva: AOI_1_1995.png
  [OK] Figura salva: AOI_1_1996.png
  [OK] Figura salva: AOI_1_1997.png
  [OK] Figura salva: AOI_1_1999.png
  [OK] Figura salva: AOI_1_2000.png
  [OK] Figura salva: AOI_1_2001.png
  [OK] Figura salva: AOI_1_2004.png
  [OK] Figura salva: AOI_1_2005.png
  [OK] Figura salva: AOI_1_2006.png
  [OK] Figura salva: AOI_1_2007.png
  [OK] Figura salva: AOI_1_2009.png
  [OK] Figura salva: AOI_1_2010.png
  [OK] Figura salva: AOI_1_2011.png
  [OK] Figura salva: AOI_1_2012.png
  [OK] Figura salva: AOI_1_2013.png
  [OK] Figura salva: AOI_1_2014.png
  [OK] Figura salva: AOI_1_2015.png
  [OK] Figura salva: AOI_1_2016.png
  [OK] Figura salva: AOI_1